# Generic catalog x LSST collection report

This notebook analyzes any target catalog that can be represented by an input table with an identifier and sky coordinates. Change the configuration cell to analyze QSO, LaStBeRu, or another CSV/FITS/Parquet catalog.

The workflow is intentionally modular:

1. Load and filter the catalog.
2. Match targets to individual visits.
3. Match targets to coadds and sample coadd property maps.
4. Write matching tables, distributions, sky maps, cutouts, and a PDF report.

Coverage and property-map queries are cached. Cutouts are disabled by default. All plot labels and report text are in English.

## 1. Configuration

Edit this cell only when changing catalogs or output locations. CATEGORY_COLUMN=None is valid for catalogs without categories. For LaStBeRu, use CATEGORY_COLUMN="extra_info" and CATEGORY_PARSER="first_extra_value"; for QSO, use the actual status column and leave the parser as "identity".

In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

from target_selection import images, match, report, target_io
from target_selection import open_collection

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_release_config.py").exists()), Path.cwd())
SOURCE_PATH = REPO_ROOT / "data" / "LaStBeRu_data" / "LaStBeRu_watchlist_LSST_grades.csv"  # Change this for QSO or another catalog.
SOURCE_NAME = "lastberu_watchlist"
ID_COLUMN, RA_COLUMN, DEC_COLUMN = "object_id", "ra", "dec"
CATEGORY_COLUMN, CATEGORY_PARSER, CATEGORY_LABEL = "extra_info", "first_extra_value", "Grade"
MAGNITUDE_COLUMN = None  # Set this when the source contains a usable magnitude column.
FILTERS = {}  # Example: {"category": ["A", "B"], "magnitude": {"column": "i_mag", "maximum": 21}}
COLLECTION_NAME = "DP2"
BANDS = tuple("ugrizy")
OUTPUT_ROOT = REPO_ROOT / "notebooks" / "outputs" / f"{SOURCE_NAME.lower()}_{COLLECTION_NAME.lower()}"
CACHE_ROOT = OUTPUT_ROOT / "cache"
FITS_ROOT = OUTPUT_ROOT / "cutouts" / "fits"
PNG_ROOT = OUTPUT_ROOT / "cutouts" / "pngs"
CUTOUT_SIZE_ARCSEC = 20.0
SEARCH_RADIUS_DEG, TILE_NSIDE, TAP_WORKERS = 0.35, 64, 2
MAKE_CUTOUTS, SAMPLE_PROPERTY_MAPS = False, True
for path in (OUTPUT_ROOT, CACHE_ROOT, FITS_ROOT, PNG_ROOT): path.mkdir(parents=True, exist_ok=True)
COLLECTION = open_collection(COLLECTION_NAME)
OUTPUT_ROOT

## 2. Load and filter targets

target_io.load returns only targets that pass the configured local filters. It preserves source columns and adds the canonical target_id, ra_deg, dec_deg, source, and optional category columns.

In [ ]:
targets = target_io.load(SOURCE_PATH, source_name=SOURCE_NAME, id_column=ID_COLUMN, ra_column=RA_COLUMN, dec_column=DEC_COLUMN, category_column=CATEGORY_COLUMN, category_parser=CATEGORY_PARSER, magnitude_column=MAGNITUDE_COLUMN, filters=FILTERS, verbose=True)
print(targets.head())
print(f"Selected targets: {len(targets):,}")

## 3. Individual-visit matching

The collection API first identifies candidate spatial tiles and then performs the exact target/visit coverage query. The result is one row per target with per-band visit counts.

In [ ]:
visit_cache = CACHE_ROOT / "visit_coverage"
coverage = match.visits(targets, collection=COLLECTION, tile_nside=TILE_NSIDE, max_workers=TAP_WORKERS, timeout_seconds=7200, search_radius_deg=SEARCH_RADIUS_DEG, cache_dir=visit_cache, bands=BANDS, verbose=True)
print(f"Targets with visit coverage: {coverage['has_visit_coverage'].sum():,}/{len(coverage):,}")

## 4. Coadd matching and property maps

Coadd existence is checked directly through Butler. Property-map values are sampled at the target coordinates; they are not measured from the cutout pixels.

In [ ]:
coadd_cache = CACHE_ROOT / "direct_coadd_coverage.csv"
coverage = match.coadds(coverage, collection=COLLECTION, cache_path=coadd_cache, reuse_cache=True, bands=BANDS, verbose=True)
for band in BANDS:
    mag_col, psf_col = f"coadd_maglim_{band}", f"coadd_psf_fwhm_arcsec_{band}"
    metric_mask = pd.to_numeric(coverage[mag_col], errors="coerce").notna() & pd.to_numeric(coverage[psf_col], errors="coerce").notna() if mag_col in coverage and psf_col in coverage else pd.Series(False, index=coverage.index)
    coverage[f"has_valid_coadd_{band}"] = coverage[f"has_coadd_{band}"].fillna(False).astype(bool) & metric_mask
coverage["has_valid_coadd"] = coverage[[f"has_valid_coadd_{band}" for band in BANDS]].any(axis=1)
coverage["valid_coadd_n_bands"] = coverage[[f"has_valid_coadd_{band}" for band in BANDS]].sum(axis=1).astype(int)
if SAMPLE_PROPERTY_MAPS and coverage["has_coadd"].fillna(False).any():
    property_targets = coverage.loc[coverage["has_coadd"].fillna(False), ["target_id", "ra_deg", "dec_deg"]].copy()
    property_key = hashlib.sha256("|".join(sorted(property_targets["target_id"].astype(str))).encode()).hexdigest()[:16]
    property_cache = CACHE_ROOT / f"coadd_properties_{property_key}.csv"
    properties = pd.read_csv(property_cache) if property_cache.exists() else match.property_maps(property_targets, collection=COLLECTION, properties=("maglim", "psf_size", "exposure_time", "sky_noise"), bands=BANDS, partial_reads=True, verbose=True)
    if not property_cache.exists(): properties.to_csv(property_cache, index=False)
    coverage = coverage.drop(columns=[c for c in properties.columns if c != "target_id" and c in coverage], errors="ignore").merge(properties, on="target_id", how="left")
# Recompute validity after property-map columns have been added.
for band in BANDS:
    mag_col, psf_col = f"coadd_maglim_{band}", f"coadd_psf_fwhm_arcsec_{band}"
    metric_mask = pd.to_numeric(coverage[mag_col], errors="coerce").notna() & pd.to_numeric(coverage[psf_col], errors="coerce").notna() if mag_col in coverage and psf_col in coverage else pd.Series(False, index=coverage.index)
    coverage[f"has_valid_coadd_{band}"] = coverage[f"has_coadd_{band}"].fillna(False).astype(bool) & metric_mask
coverage["has_valid_coadd"] = coverage[[f"has_valid_coadd_{band}" for band in BANDS]].any(axis=1)
coverage["valid_coadd_n_bands"] = coverage[[f"has_valid_coadd_{band}" for band in BANDS]].sum(axis=1).astype(int)
coverage.to_csv(OUTPUT_ROOT / "coverage_and_coadds.csv", index=False)
print(f"Targets with coadds: {coverage['has_coadd'].fillna(False).sum():,}/{len(coverage):,}")
print(f"Targets with valid coadd metrics: {coverage['has_valid_coadd'].fillna(False).sum():,}/{len(coverage):,}")

## 5. Tables, distributions, and sky maps

The report functions accept any category column. When no category is available, they produce one all group.

In [ ]:
CATEGORY_FOR_REPORT = "category" if "category" in coverage else "__all__"
if CATEGORY_FOR_REPORT == "__all__": coverage["__all__"] = "all"
summary = report.match_summary(coverage, OUTPUT_ROOT, bands=BANDS, category_column=CATEGORY_FOR_REPORT, category_label=CATEGORY_LABEL if CATEGORY_FOR_REPORT != "__all__" else "Group", name=SOURCE_NAME)
report.property_distributions(coverage, OUTPUT_ROOT, bands=BANDS, category_column=CATEGORY_FOR_REPORT, category_label=CATEGORY_LABEL if CATEGORY_FOR_REPORT != "__all__" else "Group", name=SOURCE_NAME)
report.sky_maps(coverage, OUTPUT_ROOT, bands=BANDS, name=SOURCE_NAME)
print(summary)

## 6. FITS and PNG cutouts

Set MAKE_CUTOUTS=True only when cutouts are wanted. Existing files are reused. Each FITS file is a 20-arcsec square cutout, not a complete LSST image.

In [ ]:
if MAKE_CUTOUTS:
    manifest = images.cutouts(coverage, collection=COLLECTION, png_dir=PNG_ROOT, fits_dir=FITS_ROOT, bands=BANDS, size_arcsec=CUTOUT_SIZE_ARCSEC, category_column=CATEGORY_FOR_REPORT if CATEGORY_FOR_REPORT != "__all__" else None, add_rgb=False, reuse_existing=True, verbose=True)
    manifest.to_csv(OUTPUT_ROOT / "cutout_manifest.csv", index=False)
    print(f"Generated or reused {len(manifest):,} cutout targets.")
else:
    print("MAKE_CUTOUTS=False: no cutouts generated.")

## 7. PDF report

The PDF combines available sky maps, distributions, summary tables, and one page per target with available coadd cutouts.

In [ ]:
pdf_path = report.main(coverage, fits_dir=FITS_ROOT, asset_dir=OUTPUT_ROOT, output_path=OUTPUT_ROOT / f"{SOURCE_NAME}_{COLLECTION_NAME.lower()}_report.pdf", bands=BANDS, verbose=True)
print(f"Report saved to: {pdf_path}")

## Main outputs

The output directory contains the canonical coverage table, summary tables and figures, sky maps, the optional cutout manifest, and the PDF report.